<a href="https://colab.research.google.com/github/avocado-planet/06-MCP-with-LangChain/blob/main/06_mcp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP（Model Context Protocol）完全ガイド

MCPとは何か、ローカルMCPサーバの作成方法、LangChainとの統合方法を学びます。

## 目次

**Part 1: MCPの基本**
1. セットアップ
2. MCPとは何か
3. MCPの通信方式（stdio / HTTP）

**Part 2: MCPサーバを作る**
4. FastMCP で最小のサーバを作る
5. 複数ツールを持つサーバ
6. HTTP サーバとして起動する

**Part 3: LangChain から MCP を使う**
7. stdio 接続でエージェントから使う
8. MultiServerMCPClient で複数サーバに接続
9. MCP ツール + 通常ツールの混在
10. LangGraph の StateGraph で使う

---
## 1. セットアップ

In [11]:
# MCP 関連パッケージのインストール
!pip install --pre -U langchain langgraph langchain-openai -q
!pip install langchain-mcp-adapters -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 1.4 MB/s eta 0:00:00


In [16]:
import os
from getpass import getpass
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

MODEL_NAME = "gpt-5.4-nano"

print(f"モデル: {MODEL_NAME}")

モデル: gpt-5.4-nano


---
## 2. MCPとは何か

**Model Context Protocol (MCP)** は、LLM アプリケーションが外部ツールやデータソースに
接続するための **オープンなプロトコル（通信規格）** です。

### USB-C のようなもの

```
MCP がない世界:
  LLM ──カスタムコード──→ GitHub API
  LLM ──カスタムコード──→ Slack API
  LLM ──カスタムコード──→ DB
  (それぞれ別の接続方法が必要)

MCP がある世界:
  LLM ──MCP──→ GitHub MCPサーバ
  LLM ──MCP──→ Slack MCPサーバ
  LLM ──MCP──→ DB MCPサーバ
  (全て同じプロトコルで接続)
```

### 構成要素

```
┌─────────────┐     MCP プロトコル     ┌──────────────┐
│  MCP Client │ ◄──────────────────► │  MCP Server  │
│  (LangChain │     JSON-RPC 2.0     │  (FastMCP等) │
│   エージェント)│                      │              │
└─────────────┘                      └──────────────┘
      │                                     │
  tools/list  → サーバが持つツール一覧を取得
  tools/call  → ツールを実行して結果を取得
```

## 3. MCPの通信方式

| 方式 | 説明 | 用途 |
|---|---|---|
| **stdio** | サブプロセスとして起動、標準入出力で通信 | ローカルツール、シンプルな構成 |
| **HTTP (streamable-http)** | HTTP リクエストで通信 | リモートサーバ、複数クライアント |

---
# Part 2: MCP サーバを作る

## 4. FastMCP で最小のサーバを作る

`FastMCP` を使うと、Python の関数をそのまま MCP ツールとして公開できます。
関数の docstring がツールの説明になり、LLM がいつ使うかを判断します。

In [4]:
%%writefile /tmp/math_server.py
# ============================================
# 最小の MCP サーバ（計算ツール）
# ============================================
from mcp.server.fastmcp import FastMCP

# サーバ名を指定して作成
mcp = FastMCP("Math")


@mcp.tool()
def add(a: int, b: int) -> int:
    """2つの数を足し算します"""    # ← これがLLMに見える説明
    return a + b


@mcp.tool()
def multiply(a: int, b: int) -> int:
    """2つの数を掛け算します"""    # ← これがLLMに見える説明
    return a * b


@mcp.tool()
def subtract(a: int, b: int) -> int:
    """2つの数を引き算します"""    # ← これがLLMに見える説明
    return a - b


if __name__ == "__main__":
    # SSE モードで起動（Colab の fileno() 問題を回避）
    mcp.run(transport="sse")

Writing /tmp/math_server.py


In [7]:
print("math_server.py を作成しました")
print("\nこのファイルは:")
print("  - 'Math' という名前の MCP サーバを定義")
print("  - add, multiply, subtract の3つのツールを公開")
print("  - stdio モードで起動（標準入出力で通信）")
print("\n次のセクションで LangChain から接続します")

math_server.py を作成しました

このファイルは:
  - 'Math' という名前の MCP サーバを定義
  - add, multiply, subtract の3つのツールを公開
  - stdio モードで起動（標準入出力で通信）

次のセクションで LangChain から接続します


## 5. 複数ツールを持つサーバ

もう少し実用的なサーバを作ります。文字列操作ツールのサーバです。

In [25]:
%%writefile /tmp/text_server.py
from mcp.server.fastmcp import FastMCP
import uvicorn

mcp = FastMCP("TextTools")


@mcp.tool()
def count_words(text: str) -> int:
    """テキストの単語数を数えます"""
    return len(text.split())


@mcp.tool()
def reverse_text(text: str) -> str:
    """テキストを逆順にします"""
    return text[::-1]


@mcp.tool()
def to_uppercase(text: str) -> str:
    """テキストを大文字に変換します"""
    return text.upper()


@mcp.tool()
def char_count(text: str) -> dict:
    """テキストの文字数統計を返します"""
    return {
        "total": len(text),
        "without_spaces": len(text.replace(" ", "")),
        "words": len(text.split()),
    }


if __name__ == "__main__":
    uvicorn.run(mcp.sse_app(), host="0.0.0.0", port=8001)

Overwriting /tmp/text_server.py


In [6]:
print("text_server.py を作成しました")
print("  ツール: count_words, reverse_text, to_uppercase, char_count")

text_server.py を作成しました
  ツール: count_words, reverse_text, to_uppercase, char_count


## 6. HTTP サーバとして起動する

stdio ではなく HTTP で通信するサーバも作れます。
リモートからの接続や複数クライアントに対応できます。

```python
# http_server.py
from fastmcp import FastMCP

mcp = FastMCP("WeatherService")

@mcp.tool()
def get_weather(city: str) -> str:
    """都市の天気を返します"""
    return f"{city}: 晴れ, 25度"

if __name__ == "__main__":
    # HTTP モードで起動（ポート8000）
    mcp.run(transport="streamable-http", port=8000)
```

起動: `python http_server.py`  
接続先: `http://localhost:8000/mcp`

> **Note:** Colab では HTTP サーバの起動が制限されるため、
> このノートブックでは主に stdio 方式を使います。

---
# Part 3: LangChain から MCP を使う

## 7. stdio 接続でエージェントから使う

`langchain-mcp-adapters` ライブラリを使うと、
MCP サーバのツールを LangChain のツールとして使えます。

In [19]:
import subprocess, time
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI


math_proc = subprocess.Popen(
    ["python", "/tmp/math_server.py"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(2)
print("math_server PID:", math_proc.pid if math_proc.poll() is None
      else math_proc.stderr.read().decode())

model = ChatOpenAI(model=MODEL_NAME)

'''
async def run_math_agent():
    client = MultiServerMCPClient({
        "math": {"url": "http://localhost:8000/sse", "transport": "sse"}
    })
    tools = await client.get_tools()
    print(f"取得したツール: {[t.name for t in tools]}")
    agent = create_agent(model, tools)
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "(3 + 5) x 12 を計算して"}]}
    )
    print(f"応答: {result['messages'][-1].content}")
    return result
'''

#toolの呼び出しを確認する
async def run_math_agent():
    client = MultiServerMCPClient({
        "math": {"url": "http://localhost:8000/sse", "transport": "sse"}
    })
    tools = await client.get_tools()

    agent = create_agent(model, tools)

    # stream で途中経過を表示
    async for chunk in agent.astream(
        {"messages": [{"role": "user", "content": "(3 + 5) × 12 を計算して"}]}
    ):
        for node, value in chunk.items():
            print(f"\n--- {node} ---")
            for msg in value.get("messages", []):
                print(f"  type : {msg.__class__.__name__}")
                print(f"  content: {msg.content}")
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"  tool_call: {tc['name']}({tc['args']})")
                if hasattr(msg, "name") and msg.name:
                    print(f"  tool_name: {msg.name}")

print("=" * 60)
print("SSE 接続で math_server を使う")
print("=" * 60)
result = await run_math_agent()

math_server PID: 8792
SSE 接続で math_server を使う

--- model ---
  type : AIMessage
  content: 
  tool_call: add({'a': 3, 'b': 5})

--- tools ---
  type : ToolMessage
  content: [{'type': 'text', 'text': '8', 'id': 'lc_ebe2348f-7012-48d0-8ed3-3ce1d0a7c808'}]
  tool_name: add

--- model ---
  type : AIMessage
  content: 
  tool_call: multiply({'a': 8, 'b': 12})

--- tools ---
  type : ToolMessage
  content: [{'type': 'text', 'text': '96', 'id': 'lc_d6030d19-4832-4f32-a5d6-13ea44b204b1'}]
  tool_name: multiply

--- model ---
  type : AIMessage
  content: \((3 + 5) \times 12 = 96\) です。


## 8. MultiServerMCPClient で複数サーバに接続

`MultiServerMCPClient` を使うと、複数の MCP サーバに同時接続して
全てのツールを1つのエージェントで使えます。

In [26]:
import subprocess, time

try:
    text_proc.terminate()
    time.sleep(1)
except:
    pass

text_proc = subprocess.Popen(
    ["python", "/tmp/text_server.py"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(3)
print("text_server PID:", text_proc.pid if text_proc.poll() is None
      else text_proc.stderr.read().decode())


async def run_multi_server_agent():
    client = MultiServerMCPClient({
        "math": {"url": "http://localhost:8000/sse", "transport": "sse"},
        "text": {"url": "http://localhost:8001/sse", "transport": "sse"},
    })
    tools = await client.get_tools()
    print(f"全ツール: {[t.name for t in tools]}")
    agent = create_agent(model, tools)

    print("--- 計算タスク ---")
    r1 = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "7 x 8 を計算して"}]}
    )
    print(f"応答: {r1['messages'][-1].content}")

    print("--- テキストタスク ---")
    r2 = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "Hello World を大文字にして文字数も教えて"}]}
    )
    print(f"応答: {r2['messages'][-1].content}")


print("=" * 60)
print("MultiServerMCPClient で複数サーバ接続")
print("=" * 60)
await run_multi_server_agent()

text_server PID: 13061
MultiServerMCPClient で複数サーバ接続
全ツール: ['add', 'multiply', 'subtract', 'count_words', 'reverse_text', 'to_uppercase', 'char_count']
--- 計算タスク ---
応答: 7 × 8 = **56** です。
--- テキストタスク ---
応答: - 大文字：**HELLO WORLD**
- 文字数：**11文字**（スペース含む。スペースなしだと **10文字**）


## 9. MCP ツール + 通常ツールの混在

MCP から取得したツールと、通常の `@tool` デコレータで定義したツールは
同じリストに混ぜて使えます。

In [28]:
from langchain_core.tools import tool
import random


@tool
def get_weather(city: str) -> str:
    """指定した都市の天気を返します"""
    weathers = ["晴れ", "曇り", "雨"]
    return f"{city}: {random.choice(weathers)}, {random.randint(15, 30)}度"


@tool
def get_time() -> str:
    """現在時刻を返します"""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


async def run_mixed_tools_agent():
    client = MultiServerMCPClient({
        "math": {"url": "http://localhost:8000/sse", "transport": "sse"},
    })
    mcp_tools = await client.get_tools()
    all_tools = [get_weather, get_time] + mcp_tools
    print(f"全ツール: {[t.name for t in all_tools]}")
    agent = create_agent(model, all_tools)
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "東京の天気と、今何時かと、15 x 23 の計算結果を教えて"}]}
    )
    print(f"応答: {result['messages'][-1].content}")


print("=" * 60)
print("MCP ツール + 通常ツール 混在")
print("=" * 60)
await run_mixed_tools_agent()

MCP ツール + 通常ツール 混在
全ツール: ['get_weather', 'get_time', 'add', 'multiply', 'subtract']
応答: - 東京の天気：晴れ、25度  
- 今の時刻：2026-04-09 06:00:26  
- 15 × 23 = 345


## 10. LangGraph の StateGraph で使う

`create_agent` を使わず、LangGraph の `StateGraph` で
MCP ツールを組み込むこともできます。
より細かい制御が必要な場合に便利です。

In [29]:
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import HumanMessage


async def run_langgraph_mcp():
    client = MultiServerMCPClient({
        "math": {"url": "http://localhost:8000/sse", "transport": "sse"},
    })
    tools = await client.get_tools()
    model_with_tools = model.bind_tools(tools)

    def call_model(state: MessagesState):
        response = model_with_tools.invoke(state["messages"])
        return {"messages": response}

    builder = StateGraph(MessagesState)
    builder.add_node("model", call_model)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "model")
    builder.add_conditional_edges("model", tools_condition)
    builder.add_edge("tools", "model")
    graph = builder.compile()

    result = await graph.ainvoke(
        {"messages": [HumanMessage(content="100 + 200 を計算して")]}
    )
    print(f"応答: {result['messages'][-1].content}")
    print("グラフ構造: START → model → tools → model → END")


print("=" * 60)
print("LangGraph StateGraph + MCP")
print("=" * 60)
await run_langgraph_mcp()

LangGraph StateGraph + MCP
応答: 100 + 200 = **300** です。
グラフ構造: START → model → tools → model → END


---
## おまけ: MCP サーバの実用例

### ファイル操作サーバ

In [33]:
%%writefile /tmp/file_server.py
from mcp.server.fastmcp import FastMCP
import uvicorn, os

mcp = FastMCP("FileTools")

ALLOWED_DIR = "/tmp/mcp_workspace"
os.makedirs(ALLOWED_DIR, exist_ok=True)


@mcp.tool()
def list_files() -> list[str]:
    """ワークスペース内のファイル一覧を返します"""
    return os.listdir(ALLOWED_DIR)


@mcp.tool()
def read_file(filename: str) -> str:
    """ファイルの内容を読み取ります"""
    path = os.path.join(ALLOWED_DIR, filename)
    if not os.path.exists(path):
        return f"エラー: {filename} が見つかりません"
    with open(path, "r") as f:
        return f.read()


@mcp.tool()
def write_file(filename: str, content: str) -> str:
    """ファイルに内容を書き込みます"""
    path = os.path.join(ALLOWED_DIR, filename)
    with open(path, "w") as f:
        f.write(content)
    return f"{filename} に書き込みました ({len(content)}文字)"


if __name__ == "__main__":
    uvicorn.run(mcp.sse_app(), host="0.0.0.0", port=8002)

Overwriting /tmp/file_server.py


In [35]:
import subprocess, time

#既存の file_server プロセスを停止しています。
try:
    file_proc.terminate()
    time.sleep(1)
except:
    pass

file_proc = subprocess.Popen(
    ["python", "/tmp/file_server.py"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(3)
print("file_server PID:", file_proc.pid if file_proc.poll() is None
      else file_proc.stderr.read().decode())


async def run_file_agent():
    client = MultiServerMCPClient({
        "files": {"url": "http://localhost:8002/sse", "transport": "sse"},
    })
    tools = await client.get_tools()
    print(f"ツール: {[t.name for t in tools]}")
    agent = create_agent(model, tools)
    r = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "hello.txt に Hello from MCP! と書き込んで、その後ファイル一覧を見せて"}]}
    )
    print(f"応答: {r['messages'][-1].content}")


print("=" * 60)
print("ファイル操作 MCP サーバ")
print("=" * 60)
await run_file_agent()

file_server PID: 14004
ファイル操作 MCP サーバ
ツール: ['list_files', 'read_file', 'write_file']
応答: hello.txt に **「Hello from MCP!」**を書き込みました。

その後のファイル一覧：
- hello.txt


In [38]:
cat /tmp/mcp_workspace/hello.txt

Hello from MCP!

---
## まとめ

| 項目 | 内容 |
|---|---|
| MCP とは | LLM がツール/データに接続するためのオープンプロトコル |
| FastMCP | Python 関数を MCP ツールとして公開するライブラリ |
| langchain-mcp-adapters | MCP ツールを LangChain ツールに変換するアダプタ |
| stdio | サブプロセスとして通信。ローカル用 |
| HTTP | HTTPリクエストで通信。リモート用 |
| MultiServerMCPClient | 複数 MCP サーバに同時接続 |
| ツール混在 | MCP ツール + 通常 `@tool` を同じリストで使える |